In [ ]:
import json
import math
import time
from pathlib import Path
import pandas as pd
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
DOCS_DIR        = "../../data/parsed_json/"
CHROMA_BASE     = "../../data/chroma_db_model"  # 모델별로 __{model} 접미사 붙음
GOLDEN_SET_PATH = "../../data/golden_set.json"
COLLECTION_NAME = "rfp_docs"
MAX_CHUNK_SIZE  = 1000
BATCH_SIZE      = 500

# 실험할 모델 목록 (돌리고 싶은 것만 주석 해제)
MODELS_TO_RUN = [
    "bge-m3",
    # "ko-sroberta-multitask",
    # "text-embedding-3-small",
    # "text-embedding-3-large",
    # "embed-multilingual-v3",
]

# API 키 (필요한 모델만 설정)
# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["COHERE_API_KEY"] = "..."

MODEL_COST = {
    "text-embedding-3-small": 0.02,
    "text-embedding-3-large": 0.13,
    "bge-m3":                 0.0,
    "ko-sroberta-multitask":  0.0,
    "embed-multilingual-v3":  0.10,
}

1. 유틸리티

In [ ]:
def clean(val):
    # NaN / None → 빈 문자열 (Chroma 메타데이터는 NaN·None 불가)
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [ ]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    # 청크 메타데이터(페이로드) 생성
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }

2. 청킹

In [ ]:
def chunk_section(section: dict) -> list[dict]:

    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue
        if block["type"] == "table":
            # 쌓인 텍스트 먼저 방출
            if current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = ""
                last_text_block = None
            # 표 단독 방출
            results.append({
                "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                "block":   block,
            })
        else:
            # 텍스트 블록 누적
            if len(current_text) + len(block["content"]) > MAX_CHUNK_SIZE and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    # 남은 텍스트 방출
    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })

    return results

In [ ]:
def process_doc(doc: dict) -> tuple[list[str], list[dict]]:
    # JSON 문서 1개 → (청크 텍스트 리스트, 메타데이터 리스트)
    texts, metas = [], []

    warnings = doc.get("qa", {}).get("extraction_warnings", [])
    if warnings:
        print(f"  [WARN] {doc['doc_id']} — extraction_warnings: {warnings}")

    meta = doc.get("metadata", {})
    summary  = str(clean(meta.get("사업요약")))
    사업명   = str(clean(meta.get("사업명")))
    발주기관 = str(clean(meta.get("발주기관")))

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            prefix = (
                f"[사업명] {사업명}\n"
                f"[발주기관] {발주기관}\n"
                f"[요약] {summary}\n\n"
            )
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

3. 임베딩 모델

In [ ]:
def load_embedding_model(name: str):
    if name == "text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif name == "text-embedding-3-large":
        return OpenAIEmbeddings(model="text-embedding-3-large")
    elif name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "ko-sroberta-multitask":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "embed-multilingual-v3":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model="embed-multilingual-v3.0")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")

4. Vector DB / Retrieval

In [ ]:
def save_vectorstore_for_model(model_name: str, all_texts: list, all_metas: list):
    safe_name = model_name.replace("/", "_")
    chroma_dir = f"{CHROMA_BASE}__{safe_name}"

    # 이미 저장되어 있으면 스킵
    if Path(chroma_dir).exists():
        print(f"[SKIP] {model_name} — 이미 존재: {chroma_dir}")
        return

    print(f"\n[INDEX] {model_name}")
    embedding_model = load_embedding_model(model_name)
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=chroma_dir,
    )
    for start in range(0, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vectorstore.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"  완료 ({vectorstore._collection.count()}개 청크) → {chroma_dir}")

5. 실행

In [ ]:
docs_dir = Path(DOCS_DIR)
json_files = sorted(docs_dir.glob("*.json"))
print(f"JSON 파일 수: {len(json_files)}")

all_texts, all_metas = [], []
for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        doc = json.load(f)
    texts, metas = process_doc(doc)
    all_texts.extend(texts)
    all_metas.extend(metas)

print(f"총 청크 수: {len(all_texts)}")

for model in MODELS_TO_RUN:
    save_vectorstore_for_model(model, all_texts, all_metas)

6. 검색 테스트

In [ ]:
TEST_QUERIES = [
    "보안 요구사항이 있는 사업은?",
    "사업 예산이 가장 큰 사업은?",
    "클라우드 전환 관련 사업은?",
]
VISUAL_MODEL = "bge-m3"   # 확인할 모델
TOP_K = 3

safe_name = VISUAL_MODEL.replace("/", "_")
chroma_dir = f"{CHROMA_BASE}__{safe_name}"
vs = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=load_embedding_model(VISUAL_MODEL),
    persist_directory=chroma_dir,
)

for q in TEST_QUERIES:
    print(f"\n[질의] {q}")
    results = vs.similarity_search_with_relevance_scores(q, k=TOP_K)
    for i, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        print(f"  [{i}] score={score:.4f} | {m.get('사업명')} | {m.get('header_path')}")
        print(f"       {doc.page_content[:200]}...")

7. 평가 (Recall@k / MRR)

In [ ]:
def get_retrieved_ids(vectorstore: Chroma, question: str, k: int) -> list:
    results = vectorstore.similarity_search_with_relevance_scores(question, k=k)
    return [
        f"{doc.metadata.get('doc_id', '')}__{doc.metadata.get('block_id', '')}"
        for doc, _ in results
    ]


def recall_at_k(retrieved_ids, answer_id, k):
    return 1.0 if answer_id in retrieved_ids[:k] else 0.0


def mrr(retrieved_ids, answer_id):
    for rank, cid in enumerate(retrieved_ids, 1):
        if cid == answer_id:
            return 1.0 / rank
    return 0.0


def evaluate_model(model_name: str, golden_set: list) -> dict:
    safe_name = model_name.replace("/", "_")
    chroma_dir = f"{CHROMA_BASE}__{safe_name}"

    print(f"\n[EVAL] {model_name}")
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=load_embedding_model(model_name),
        persist_directory=chroma_dir,
    )

    recall3, recall5, recall10, mrr_scores, query_times = [], [], [], [], []

    for item in golden_set:
        question  = item["question"]
        answer_id = item["answer_chunk_id"]

        t0 = time.time()
        retrieved = get_retrieved_ids(vectorstore, question, k=10)
        query_times.append(time.time() - t0)

        recall3.append(recall_at_k(retrieved, answer_id, 3))
        recall5.append(recall_at_k(retrieved, answer_id, 5))
        recall10.append(recall_at_k(retrieved, answer_id, 10))
        mrr_scores.append(mrr(retrieved, answer_id))

    n = len(golden_set)
    return {
        "model":        model_name,
        "Recall@3":     round(sum(recall3)    / n, 4),
        "Recall@5":     round(sum(recall5)    / n, 4),
        "Recall@10":    round(sum(recall10)   / n, 4),
        "MRR":          round(sum(mrr_scores) / n, 4),
        "avg_query_ms": round(sum(query_times) / n * 1000, 1),
        "비용/1M":      f"${MODEL_COST.get(model_name, 0)}",
    }

In [ ]:
if not Path(GOLDEN_SET_PATH).exists():
    print(f"[SKIP] 골든셋 파일 없음: {GOLDEN_SET_PATH}")
    print("PM과 함께 golden_set.json을 먼저 만들어 주세요.")
else:
    with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
        golden_set = json.load(f)
    print(f"골든셋 문항 수: {len(golden_set)}")

    results = []
    for model in MODELS_TO_RUN:
        try:
            results.append(evaluate_model(model, golden_set))
        except Exception as e:
            print(f"  [ERROR] {model}: {e}")

    df = pd.DataFrame(results).set_index("model")
    display(df)
    df.to_csv("../../data/eval_model_results.csv", encoding="utf-8-sig")
    print("\n결과 저장: data/eval_model_results.csv")

## 임베딩 모델 비교 결과

| 모델 | Recall@3 | Recall@5 | Recall@10 | MRR | avg_query_ms | 비용/1M |
|------|----------|----------|-----------|-----|--------------|----------|
| bge-m3 | - | - | - | - | - | $0 |
| ko-sroberta-multitask | - | - | - | - | - | $0 |
| text-embedding-3-small | - | - | - | - | - | $0.02 |
| text-embedding-3-large | - | - | - | - | - | $0.13 |
| embed-multilingual-v3 | - | - | - | - | - | $0.10 |

**최종 선택 모델**: (실험 후 기입)  
**선택 근거**: (실험 후 기입)